In [2]:
import pandas as pd
import numpy as np
import sys
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, List
import time
import argparse

import config
from src.data_engine import DataRepository
from src.strategy import SEPAStrategy
from src.database import DatabaseManager
from src.features import FeatureEngineer
from src.utils import get_latest_trading_day
import logging

# Setup logging
logging.basicConfig(
    level=getattr(logging, config.LOG_LEVEL),
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [3]:
scan_date='2025-12-15'
csv_output=None
use_ml=True
model_path=None
tickers=None
debug=False
skip_cache_update=True

In [4]:
ml_scorer = None
if use_ml:
    from src.ml_scorer import MLScorer

    if model_path is None:
        model_path = 'models/model_prod.json'

    try:
        ml_scorer = MLScorer(model_path=model_path, log_predictions=True)
        # Extract just filename from path
        import os
        model_name = os.path.basename(model_path)
        print(f"\n[ML] Loaded model: {model_name}")
        print(f"[ML] Model version: {ml_scorer.model_version}")
        print(f"[ML] Features required: {len(ml_scorer.feature_names)}")
        print(f"[ML] Scoring mode: Informational (all SEPA signals retained)")
    except Exception as e:
        print(f"\n[WARN] ML model loading failed: {e}")
        print("        Proceeding without ML scoring...\n")
        use_ml = False
        ml_scorer = None

# Determine scan date
if scan_date:
    scan_date_obj = pd.Timestamp(scan_date)
else:
    # Use the most recent completed trading day
    scan_date_obj = get_latest_trading_day()
scan_date_str = scan_date_obj.strftime('%Y-%m-%d')

INFO:src.ml_scorer:Model loaded: models/model_prod.json
INFO:src.ml_scorer:Metadata loaded: 37 features expected
INFO:src.ml_scorer:MLScorer initialized with model: model_prod.json
INFO:src.ml_scorer:  Features required: 37
INFO:src.ml_scorer:  Model version: 2025-12-16
INFO:src.ml_scorer:  Prediction logging: ON



[ML] Loaded model: model_prod.json
[ML] Model version: 2025-12-16
[ML] Features required: 37
[ML] Scoring mode: Informational (all SEPA signals retained)


In [10]:
data_repo = DataRepository()
db = DatabaseManager()
tickers = data_repo.update_universe()
tickers.append(config.BENCHMARK_TICKER)


INFO:src.database:Database initialized at /Users/ceolwaerc/Desktop/Projects/mm-strat/database/trades.db
INFO:src.data_engine:Fetching ticker universe from source: PRICE_FOLDER
INFO:src.data_engine:Scanning data/price folder for tickers...
INFO:src.data_engine:Found 2413 tickers from price folder


In [29]:
# load price data
min_date = (pd.Timestamp.now() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')
ticker_data = data_repo.get_batch_data(
    tickers, 
    min_date=min_date, 
    check_min_date=False,
    show_progress=True,
    required_end_date='2025-12-04',
    force_cache_only=True
)

Loading price data: 100%|██████████| 2414/2414 [00:02<00:00, 1078.04ticker/s]


In [ ]:
# SEPA filtering
benchmark_data = data_repo.get_benchmark_data(check_min_date=False,required_end_date=scan_date,force_cache_only=True)
feature_engine = FeatureEngineer(benchmark_data=benchmark_data)
strategy = SEPAStrategy(benchmark_data=benchmark_data)
enriched_data = feature_engine.process_universe_batch(ticker_data)
results = strategy.batch_scan_universe(enriched_data, scan_date=scan_date_obj)

INFO:src.features:FeatureEngineer initialized in dual-stage mode
Computing Features: 100%|██████████| 2414/2414 [00:37<00:00, 63.97ticker/s]
INFO:src.features:Batch processed 2414/2414 tickers successfully
INFO:src.vectorized_screening:Batch screen: 626 trend OK, 68 breakout OK, 12 new triggers
INFO:src.strategy:Batch scan: 626 trend OK, 20 breakout OK, 12 new triggers


In [37]:
trend_ok_stocks = results.get('trend_ok_stocks', [])
breakout_stocks = results.get('breakout_stocks', [])
qualifying_stocks = results.get('qualifying_stocks', [])
new_triggers_today = results['new_triggers']


In [40]:
current_buy_list = db.get_buy_list(active_only=True, as_of_date=scan_date_str)
tickers_in_buy_list = set(current_buy_list['ticker'].tolist()) if not current_buy_list.empty else set()
trend_ok_tickers = set([s['ticker'] for s in trend_ok_stocks])

tickers_to_add = set([t['ticker'] for t in new_triggers_today if t['ticker'] not in tickers_in_buy_list])
tickers_to_remove = tickers_in_buy_list - trend_ok_tickers
active_tickers = (tickers_to_add | tickers_in_buy_list) - tickers_to_remove

In [ ]:
# Execute Removals
for ticker in tickers_to_remove:
    db.remove_from_buy_list(ticker, reason='trend_broken')
    db.log_buy_list_activity(
        ticker=ticker,
        action='REMOVED',
        action_date=scan_date_str,
        reason='trend_broken'
    )

In [46]:
# Execute Additions
for trigger in new_triggers_today:
    ticker = trigger['ticker']
    signal_price = trigger['entry_price']  # Close price on signal date

    # Extract indicator values from enriched data
    ticker_df = enriched_data.get(ticker)

    # Get indicator values - use scan_date if available, otherwise use last available
    if ticker_df is not None:
        if scan_date_obj in ticker_df.index:
            row = ticker_df.loc[scan_date_obj]
        else:
            # Use last available date before scan_date
            available_dates = ticker_df.index[ticker_df.index <= scan_date_obj]
            if len(available_dates) > 0:
                row = ticker_df.loc[available_dates[-1]]
            else:
                row = None

        if row is not None:
            ma50 = row.get('SMA_50')
            ma150 = row.get('SMA_150')
            ma200 = row.get('SMA_200')
            high_52w = row.get('High_52W')
            low_52w = row.get('Low_52W')
        else:
            ma50 = ma150 = ma200 = high_52w = low_52w = None
    else:
        ma50 = ma150 = ma200 = high_52w = low_52w = None

    # Get ML scores and features if available
    ml_prob = None
    ml_rank = None
    ml_model_ver = None
    ml_score_date = None
    ml_features_dict = None

    db.add_to_buy_list(
        ticker=ticker,
        signal_date=scan_date_str,
        signal_price=signal_price,
        current_price=signal_price,  # Same as signal_price initially
        rs=trigger.get('rs'),
        vol_ratio=trigger.get('vol_ratio'),
        # Indicator values
        ma50=ma50,
        ma150=ma150,
        ma200=ma200,
        high_52w=high_52w,
        low_52w=low_52w,
        # ML scores and features
        ml_probability=ml_prob,
        ml_rank=ml_rank,
        ml_model_version=ml_model_ver,
        ml_score_date=ml_score_date,
        ml_features=ml_features_dict
    )
    db.log_buy_list_activity(
        ticker=ticker,
        action='ADDED',
        action_date=scan_date_str,
        reason='new_trigger',
        entry_price=trigger['entry_price'],
        stop_price=trigger.get('stop_price'),
        target_price=trigger.get('target_price'),
        rs=trigger.get('rs'),
        vol_ratio=trigger.get('vol_ratio')
    )


In [47]:
# Update existing tickers (those that remain in buy_list)
tickers_to_update = tickers_in_buy_list & trend_ok_tickers  # Intersection: in buy_list AND still trend OK
for ticker in tickers_to_update:
    # Find the stock in trend_ok_stocks
    stock_data = next((s for s in trend_ok_stocks if s['ticker'] == ticker), None)
    if stock_data:
        # Extract updated indicator values
        ticker_df = enriched_data.get(ticker)
        
        # Get indicator values - use scan_date if available, otherwise use last available
        actual_update_date = scan_date_str
        if ticker_df is not None:
            if scan_date_obj in ticker_df.index:
                row = ticker_df.loc[scan_date_obj]
            else:
                # Use last available date before scan_date
                available_dates = ticker_df.index[ticker_df.index <= scan_date_obj]
                if len(available_dates) > 0:
                    last_date = available_dates[-1]
                    row = ticker_df.loc[last_date]
                    actual_update_date = last_date.strftime('%Y-%m-%d')  # Update to actual data date
                else:
                    row = None
            
            if row is not None:
                ma50 = row.get('SMA_50')
                ma150 = row.get('SMA_150')
                ma200 = row.get('SMA_200')
                high_52w = row.get('High_52W')
                low_52w = row.get('Low_52W')
            else:
                ma50 = ma150 = ma200 = high_52w = low_52w = None
        else:
            ma50 = ma150 = ma200 = high_52w = low_52w = None
        
        db.update_buy_list_metrics(
            ticker=ticker,
            scan_date=actual_update_date,
            current_price=stock_data['price'],
            rs=stock_data.get('rs'),
            vol_ratio=stock_data.get('vol_ratio'),
            # Indicator values (updated)
            ma50=ma50,
            ma150=ma150,
            ma200=ma200,
            high_52w=high_52w,
            low_52w=low_52w
        )


In [92]:
print(f"       Preparing scoring for {len(active_tickers)} existing tickers with ML...")
ml_rescore_start = time.time()
from src.fundamental_merger import FundamentalMerger
fund_merger = FundamentalMerger()

# Prepare features for existing tickers
ml_rescore_candidates = []
for ticker in active_tickers:
    ticker_df = enriched_data.get(ticker)
    
    if ticker_df is None or len(ticker_df) == 0:
        continue
    
    # get heavyweight features
    ticker_df = feature_engine.calculate_heavyweight_features(ticker_df, ticker)

    # Get latest row (or row at scan_date)
    if scan_date_obj in ticker_df.index:
        row_date = scan_date_obj
        row = ticker_df.loc[scan_date_obj]
    else:
        # Use last available date
        available_dates = ticker_df.index[ticker_df.index <= scan_date_obj]
        if len(available_dates) > 0:
            row_date = available_dates[-1]
            row = ticker_df.loc[row_date]
        else:
            continue
    
    # Get fundamental data
    single_date_df = pd.DataFrame({
        'Date': [row_date],
        'Close': [row.get('Close', np.nan)]
    }).set_index('Date')

    
    try:
        merged_df = fund_merger.merge_ticker_data(ticker, single_date_df)
        fund_cols = [c for c in merged_df.columns if c not in ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Adj Close']]
        fund_data = merged_df[fund_cols].iloc[0] if len(merged_df) > 0 else None
    except Exception as e:
        logger.warning(f"       [{ticker}] Failed to load fundamentals for re-scoring: {e}")
        fund_data = None
    
    # Merge technical + fundamental features
    candidate_features = {
        'ticker': ticker,
        'date': scan_date_obj,
        **row.to_dict(),
    }
    
    if fund_data is not None:
        candidate_features.update(fund_data.to_dict())
    
    ml_rescore_candidates.append(candidate_features)



       Preparing scoring for 12 existing tickers with ML...


In [93]:
ml_rescore_candidates[0]

{'ticker': 'BPOP',
 'date': Timestamp('2025-12-15 00:00:00'),
 'Open': 122.18,
 'High': 122.98,
 'Low': 120.92,
 'Close': 122.58,
 'Volume': 1420000,
 'SMA_50': 116.27699999999999,
 'Price_vs_SMA_50': 5.420676488041497,
 'SMA_150': 115.4172,
 'Price_vs_SMA_150': 6.206007423503607,
 'SMA_200': 109.44239999999999,
 'Price_vs_SMA_200': 12.0041227166071,
 'ATR': 2.654285714285713,
 'High_52W': 128.04,
 'Low_52W': 81.63,
 'Vol_MA': 591107.66,
 'Vol_Ratio': 2.402269664378905,
 'High_20D': 121.07,
 'Breakout': True,
 'RS': 0.18007139394473579,
 'RS_MA': 0.17617607095584759,
 'nATR': 2.1653497424422525,
 'VCP_Ratio': 0.8650302916744737,
 'Consolidation_Width': 11.396638929678577,
 'Dry_Up_Volume': 1.3732868899042858,
 'RSI_14': 78.75739644970417,
 'RSI_Regime': 1,
 'Dist_From_52W_High': -4.264292408622301,
 'Is_Green_Day': 1,
 'Green_Days_Ratio_20D': 0.65,
 'SMA_50_Slope': -0.13996723413348508,
 'nATR_Lag1': 2.2147753955798875,
 'ATR_Lag1': 2.6814285714285697,
 'VCP_Ratio_Lag1': 0.906111492602

In [99]:
# Score existing tickers
if len(ml_rescore_candidates) > 0:
    candidates_df = pd.DataFrame(ml_rescore_candidates)
    
    probabilities, ranks = ml_scorer.score_batch(
        candidates_df
    )
    
    # Update ML scores in database (ranks will be recalculated in Step 8)
    for idx, ticker in enumerate(candidates_df['ticker']):
        ml_prob = float(probabilities[idx])
        ml_rank = int(ranks[idx])
        
        # Get existing entry to preserve other data
        buy_list_df = db.get_buy_list(active_only=True, as_of_date=scan_date_str)
        ticker_entry = buy_list_df[buy_list_df['ticker'] == ticker]
        
        if not ticker_entry.empty:
            # Update with ML scores
            import json
            ml_features_dict = candidates_df.iloc[idx].drop(['ticker', 'date']).to_dict()
            
            db.update_buy_list_metrics(
                ticker=ticker,
                scan_date=scan_date_str,
                current_price=ticker_entry.iloc[0]['current_price'],
                rs=ticker_entry.iloc[0].get('rs'),
                vol_ratio=ticker_entry.iloc[0].get('volume_ratio'),
                ma50=ticker_entry.iloc[0].get('ma50'),
                ma150=ticker_entry.iloc[0].get('ma150'),
                ma200=ticker_entry.iloc[0].get('ma200'),
                high_52w=ticker_entry.iloc[0].get('high_52w'),
                low_52w=ticker_entry.iloc[0].get('low_52w'),
                ml_probability=ml_prob,
                ml_rank=ml_rank,
                ml_model_version=ml_scorer.model_version,
                ml_score_date=scan_date_str,
                ml_features=json.dumps(ml_features_dict,default=str)
            )
    # calculate and sort by ml rank
    ml_rescore_time = time.time() - ml_rescore_start
    print(f"       Re-scored {len(candidates_df)} tickers in {ml_rescore_time:.1f}s")
    logger.info(f"ML re-scoring took {ml_rescore_time:.1f}s for {len(candidates_df)} tickers")
        


INFO:src.ml_scorer:Scored 12 candidates: prob range [0.025, 0.737]
INFO:__main__:ML re-scoring took 114.7s for 12 tickers


       Re-scored 12 tickers in 114.7s


In [100]:
buy_list_df = db.get_buy_list(active_only=True, as_of_date=scan_date_str)
buy_list_df

,ticker,signal_date,signal_price,current_price,entry_price,stop_price,target_price,atr,rs,volume_ratio,...,rsi14_lag1,dist_from_52w_high_lag1,ml_probability,ml_rank,ml_model_version,ml_score_date,ml_features,last_updated,status,notes
0,GM,2025-12-15,81.98,81.98,None,None,None,None,0.120430,1.451058,...,None,None,0.132415,9,2025-12-16,2025-12-15,"{'Open': 80.65, 'High': 82.02, 'Low': 80.58, '...",2025-12-15,active,None
1,INNV,2025-12-15,5.82,5.82,None,None,None,None,0.008550,1.789483,...,None,None,0.663039,2,2025-12-16,2025-12-15,"{'Open': 5.42, 'High': 5.87, 'Low': 5.36, 'Clo...",2025-12-15,active,None
2,AEO,2025-12-15,27.01,27.01,None,None,None,None,0.039678,1.853221,...,None,None,0.736562,1,2025-12-16,2025-12-15,"{'Open': 25.42, 'High': 27.31, 'Low': 25.37, '...",2025-12-15,active,None
3,PANL,2025-12-15,7.39,7.39,None,None,None,None,0.010856,1.208330,...,None,None,0.384020,5,2025-12-16,2025-12-15,"{'Open': 7.19, 'High': 7.4, 'Low': 7.18, 'Clos...",2025-12-15,active,None
4,BPOP,2025-12-15,122.58,122.58,None,None,None,None,0.180071,2.402270,...,None,None,0.410859,4,2025-12-16,2025-12-15,"{'Open': 122.18, 'High': 122.98, 'Low': 120.92...",2025-12-15,active,None
5,CWBC,2025-12-15,24.20,24.20,None,None,None,None,0.035550,1.828140,...,None,None,0.153963,7,2025-12-16,2025-12-15,"{'Open': 23.99, 'High': 24.3, 'Low': 23.9, 'Cl...",2025-12-15,active,None
6,IMNM,2025-12-15,22.64,22.64,None,None,None,None,0.033258,8.437424,...,None,None,0.424633,3,2025-12-16,2025-12-15,"{'Open': 22.92, 'High': 25.3, 'Low': 21.22, 'C...",2025-12-15,active,None
7,HTBK,2025-12-15,12.10,12.10,None,None,None,None,0.017775,1.134148,...,None,None,0.104616,10,2025-12-16,2025-12-15,"{'Open': 12.13, 'High': 12.19, 'Low': 12.04, '...",2025-12-15,active,None
8,LEA,2025-12-15,116.07,116.07,None,None,None,None,0.170508,1.511546,...,None,None,0.375749,6,2025-12-16,2025-12-15,"{'Open': 114.73, 'High': 116.51, 'Low': 114.36...",2025-12-15,active,None
9,KAR,2025-12-15,28.84,28.84,None,None,None,None,0.042366,1.520788,...,None,None,0.062390,11,2025-12-16,2025-12-15,"{'Open': 28.78, 'High': 29.91, 'Low': 28.73, '...",2025-12-15,active,None


In [82]:
'ml_probability' in buy_list_df.columns

True

In [77]:
candidates_df['ml_score_date']

KeyError: 'ml_score_date'